<a href="https://colab.research.google.com/github/VB156/ACA-final-project/blob/main/VikaItayDataMining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import datetime
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

First time for anything, huh? This code should run only once. If it's commented, leave it commented

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

Data Analysis

In [4]:
os.chdir('/content')
churn = pd.read_csv('churn_dataset_train.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'churn_dataset_train.csv'

In [ ]:
# Okay let's make sure there are no 'weird' values in all of the columns...
for col in churn.select_dtypes(include=np.number).columns:
    print(f"{col}:")
    print(f"  Nulls: {churn[col].isnull().sum()}")
    print(f"  Min/Max: {churn[col].min()}, {churn[col].max()}")
    print(f"  Unique: {churn[col].nunique()}")
    print(f"  Outliers: {(np.abs((churn[col] - churn[col].mean()) / churn[col].std()) > 3).sum()} rows beyond 3 std devs\n")
for col in churn.select_dtypes(include='object').columns:
    print(f"{col}:")
    print(f"  Nulls: {churn[col].isnull().sum()}")
    print(f"  Unique: {churn[col].nunique()} → {churn[col].unique()[:10]}")

Unnamed: 0:
  Nulls: 0
  Min/Max: 0, 36989
  Unique: 24784
  Outliers: 0 rows beyond 3 std devs

age:
  Nulls: 0
  Min/Max: 10, 64
  Unique: 55
  Outliers: 0 rows beyond 3 std devs

days_since_last_login:
  Nulls: 0
  Min/Max: -999, 26
  Unique: 27
  Outliers: 1352 rows beyond 3 std devs

avg_time_spent:
  Nulls: 0
  Min/Max: -2281.236526, 3040.41
  Unique: 18859
  Outliers: 540 rows beyond 3 std devs

avg_transaction_value:
  Nulls: 0
  Min/Max: 800.46, 99914.05
  Unique: 24741
  Outliers: 477 rows beyond 3 std devs

points_in_wallet:
  Nulls: 2287
  Min/Max: -549.3574977, 1816.933696
  Unique: 17548
  Outliers: 398 rows beyond 3 std devs

churn:
  Nulls: 0
  Min/Max: 0, 1
  Unique: 2
  Outliers: 0 rows beyond 3 std devs

customer_id:
  Nulls: 0
  Unique: 24784 → ['fffe43004900440033003200390032003400'
 'fffe43004900440036003200310038003600' 'fffe430049004400350032003200'
 'fffe43004900440032003200380034003600'
 'fffe43004900440035003600320032003100'
 'fffe4300490044003400350030003800

Data Preprocessing

In [ ]:
# These are obviously non-informative, or they are strings which we can't really work with:
irrelevant_cols = ['Unnamed: 0', 'customer_id', 'Name', 'security_no', 'referral_id', 'past_complaint', 'complaint_status']
for c in irrelevant_cols:
  churn = churn.drop(c, axis=1)
churn.head()

,age,gender,region_category,membership_category,joining_date,joined_through_referral,preferred_offer_types,medium_of_operation,internet_option,days_since_last_login,avg_time_spent,avg_transaction_value,avg_frequency_login_days,points_in_wallet,used_special_discount,offer_application_preference,feedback,churn
0,30,F,Village,No Membership,09/08/2017,Yes,Gift Vouchers/Coupons,Desktop,Mobile_Data,20,118.390000,10579.56,24,610.360000,Yes,No,Poor Product Quality,1
1,55,M,Village,Silver Membership,09/02/2016,No,Credit/Debit Card Offers,Smartphone,Wi-Fi,15,179.420000,22963.05,27,694.650000,Yes,No,Poor Product Quality,0
2,47,M,City,Basic Membership,11/06/2017,Yes,Credit/Debit Card Offers,NaN,Wi-Fi,23,42.230000,32604.41,Error,520.620000,Yes,No,Poor Product Quality,1
3,18,M,Town,Gold Membership,09/02/2016,No,Credit/Debit Card Offers,Desktop,Mobile_Data,7,-1035.833706,48913.61,27,1150.093442,Yes,No,Too many ads,0
4,28,F,City,No Membership,16/07/2017,Yes,Credit/Debit Card Offers,Both,Fiber_Optic,10,449.770000,20010.02,14,653.040000,Yes,Yes,No reason specified,0


In [ ]:
# These require some editing:
print(f"Before: {set(churn['gender'].values)}")
churn = churn[churn['gender'] != 'Unknown'].reset_index()
print(f"After: {set(churn['gender'].values)}")
print(f"Before: {churn['joining_date'][0:5]}")
for i in range(len(churn['joining_date'])):
  churn.loc[i,'joining_date'] = abs((pd.to_datetime(churn.loc[i,'joining_date'], format='%d/%m/%Y') - datetime.datetime.today()).days)
print(f"After: {churn['joining_date'][0:5]}")
print(f"Before: {set(churn['medium_of_operation'].values)}, {set(churn['joined_through_referral'].values)}")
churn = churn.dropna()
print(f"After: {set(churn['medium_of_operation'].values)}, {set(churn['joined_through_referral'].values)}")
print(f"Before: {set(churn['avg_frequency_login_days'].values)}")
churn = churn[churn['avg_frequency_login_days'] != 'Error'].reset_index()
print(f"After: {set(churn['avg_frequency_login_days'].values)}")

Before: {'M', 'F', 'Unknown'}
After: {'M', 'F'}
Before: 0    09/08/2017
1    09/02/2016
2    11/06/2017
3    09/02/2016
4    16/07/2017
Name: joining_date, dtype: object
After: 0    2821
1    3368
2    2880
3    3368
4    2845
Name: joining_date, dtype: object
Before: {'Desktop', nan, 'Smartphone', 'Both'}, {'No', nan, 'Yes'}
After: {'Desktop', 'Smartphone', 'Both'}, {'No', 'Yes'}
Before: {'9', '37.57986594', '40.28262011', '49.36101144', '-8.68185081', '-9.683109495', '29.91835673', '50.64895275', '-5.938582549', '-10.10118954', '41.16266368', '-4.622815409', '29.16584426', '41.58210116', '4.123505593', '28.16282391', '2.49891818', '39.93003268', '-8.104517563', '43.11808015', '0.740084813', '29', '-17.07762543', '39.01260915', '-15.19906384', '49.70530762', '3', '-19.03426612', '25.91157636', '31.96578356', '-13.76815838', '-10.05103279', '33.22209825', '35.89934266', '47.74895186', '-9.014644272', '-7.384860972', '43.78725707', '15', '-7.771873798', '34.32765303', '-13.55840129', '4

In [ ]:
# These are obviously need to be transformed:
one_hot = ['gender', 'region_category', 'membership_category', 'feedback', 'used_special_discount', 'offer_application_preference', 'joined_through_referral', 'preferred_offer_types', 'medium_of_operation', 'internet_option']
for c in one_hot:
  churn = pd.concat([churn, pd.get_dummies(churn[c], prefix=c)], axis=1)
  churn = churn.drop(c, axis=1)
churn.head()

,level_0,index,age,joining_date,days_since_last_login,avg_time_spent,avg_transaction_value,avg_frequency_login_days,points_in_wallet,churn,...,joined_through_referral_Yes,preferred_offer_types_Credit/Debit Card Offers,preferred_offer_types_Gift Vouchers/Coupons,preferred_offer_types_Without Offers,medium_of_operation_Both,medium_of_operation_Desktop,medium_of_operation_Smartphone,internet_option_Fiber_Optic,internet_option_Mobile_Data,internet_option_Wi-Fi
0,0,0,30,2821,20,118.390000,10579.56,24,610.360000,1,...,True,False,True,False,False,True,False,False,True,False
1,1,1,55,3368,15,179.420000,22963.05,27,694.650000,0,...,False,True,False,False,False,False,True,False,False,True
2,3,3,18,3368,7,-1035.833706,48913.61,27,1150.093442,0,...,False,True,False,False,False,True,False,False,True,False
3,4,4,28,2845,10,449.770000,20010.02,14,653.040000,0,...,True,True,False,False,True,False,False,True,False,False
4,5,5,45,2996,14,159.170000,48040.07,13,679.070000,0,...,True,False,False,True,False,False,True,True,False,False


In [ ]:
corr = churn.corr()

KeyboardInterrupt: 